In [17]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import glob
import imageio
import mediapipe as mp
import cv2
import os
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score
from tensorflow.keras.utils import to_categorical  # For converting labels to categorical format (one-hot encoding)
from tensorflow.keras.models import Sequential  # For defining the neural network architecture
from tensorflow.keras.layers import LSTM, Dense  # For adding LSTM and Dense layers to the model
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping  # For logging training progress for TensorBoard
from tensorflow.keras.models import load_model  # For loading pre-trained models
from tensorflow.keras.metrics import F1Score
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam

In [2]:
# gloabl variables for label mapping
dynamic_label_names = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'HELLO', 'I', 'J', 'K', 'L', 'M', 'N', 'NO', 'O', 'P', 'Q', 'R', 'S', 'SORRY', 'T', 'THANKYOU', 'U', 'V', 'W', 'X', 'Y', 'YES', 'Z']
dynamic_label_map = {label: num for num, label in enumerate(dynamic_label_names)}

In [3]:
# load train, validation, and test data

def load_split(split):
    videos = []
    labels = []
    for label in dynamic_label_names:
        video_paths = sorted(glob.glob(f'datasets/SignVideos/{split}/{label}/*_frames'))
        for video in video_paths:
            frame_paths = sorted(glob.glob(f'{video}/*.jpg'))
            video_frames = []
            for path in frame_paths:
                frame = imageio.imread(path)
                video_frames.append(frame)
            videos.append(video_frames)
            labels.append(label)
    return videos, np.array(labels)

train_videos, train_labels = load_split('train')
val_videos, val_labels = load_split('validation')
test_videos, test_labels = load_split('test')

C:\Users\Ida\AppData\Local\Temp\ipykernel_38412\3014800117.py:12: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  frame = imageio.imread(path)


In [4]:
# extract landmarks from videos using MediaPipe Hands (see examples/hands.py)

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

def extract_landmarks(videos, labels):
    landmarks = []
    valid_labels = []

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5) as hands:

        for idx, video in enumerate(videos):
            video_landmarks = []
            for frame in video:
                results = hands.process(frame)
                left = np.zeros(63)
                right = np.zeros(63)

                if results.multi_hand_landmarks and results.multi_handedness:
                    for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                        coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark]).flatten()
                        if handedness.classification[0].label == 'Left':
                            left = coords
                        else:
                            right = coords
                    features = np.concatenate([left, right])
                    video_landmarks.append(features)

            # only keep videos where at least one hand was detected in at least one frame
            if len(video_landmarks) > 0:
                landmarks.append(video_landmarks)
                valid_labels.append(labels[idx])

    return landmarks, np.array(valid_labels)

train_landmarks, train_labels = extract_landmarks(train_videos, train_labels)
val_landmarks, val_labels = extract_landmarks(val_videos, val_labels)
test_landmarks, test_labels = extract_landmarks(test_videos, test_labels)

# train_labels_int = np.array([dynamic_label_map[l] for l in train_labels])
# train_labels_cat = to_categorical(train_labels_int, num_classes=len(dynamic_label_names))

# val_labels_int = np.array([dynamic_label_map[l] for l in val_labels])
# val_labels_cat = to_categorical(val_labels_int, num_classes=len(dynamic_label_names))

# test_labels_int = np.array([dynamic_label_map[l] for l in test_labels])
# test_labels_cat = to_categorical(test_labels_int, num_classes=len(dynamic_label_names))

In [5]:
# Find max video length
max_length = max(max(len(seq) for seq in train_landmarks), 
                 max(len(seq) for seq in val_landmarks),
                 max(len(seq) for seq in test_landmarks))

# Pad all video
train_landmarks_padded = pad_sequences(train_landmarks, maxlen=max_length, padding='post', dtype='float32')
val_landmarks_padded = pad_sequences(val_landmarks, maxlen=max_length, padding='post', dtype='float32')
test_landmarks_padded = pad_sequences(test_landmarks, maxlen=max_length, padding='post', dtype='float32')

# Convert labels to categorical
train_labels_int = np.array([dynamic_label_map[l] for l in train_labels])
train_labels_cat = to_categorical(train_labels_int, num_classes=len(dynamic_label_names))

val_labels_int = np.array([dynamic_label_map[l] for l in val_labels])
val_labels_cat = to_categorical(val_labels_int, num_classes=len(dynamic_label_names))

test_labels_int = np.array([dynamic_label_map[l] for l in test_labels])
test_labels_cat = to_categorical(test_labels_int, num_classes=len(dynamic_label_names))

In [30]:
# define LSTM model for dynamic sign classification
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(max_length, 126)))
model.add(LSTM(128, return_sequences=False, activation='relu'))
# model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(len(dynamic_label_names), activation='softmax'))

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=[F1Score(average='macro')])

log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)
early_stopping = EarlyStopping(monitor='val_f1_score', 
                               patience=3, 
                               restore_best_weights=True)

# start training the model using padded sequences
model.fit(train_landmarks_padded, train_labels_cat, 
          validation_data=(val_landmarks_padded, val_labels_cat),
          epochs=200, callbacks=[tb_callback, early_stopping])

Epoch 1/200


C:\Users\Ida\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 294ms/step - f1_score: 0.0390 - loss: 3.4326 - val_f1_score: 0.0182 - val_loss: 3.4080
Epoch 2/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - f1_score: 0.0274 - loss: 2902.5886 - val_f1_score: 0.0000e+00 - val_loss: 4604.2759
Epoch 3/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - f1_score: 0.0087 - loss: 4829.7827 - val_f1_score: 0.0020 - val_loss: 3.4338
Epoch 4/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - f1_score: 0.0042 - loss: 3.4325 - val_f1_score: 0.0000e+00 - val_loss: 3.4336


In [31]:
# evaluate the model on the test set
# test_images, test_labels = load_split('test')

# test_landmarks, test_labels = extract_landmarks(test_images, test_labels)
# test_labels_int = np.array([dynamic_label_map[l] for l in test_labels])
# test_labels_cat = to_categorical(test_labels_int, num_classes=len(dynamic_label_names))

test_loss, test_acc = model.evaluate(test_landmarks_padded, test_labels_cat)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - f1_score: 0.0342 - loss: 3.4262 
Test Loss: 3.4262
Test Accuracy: 0.0342


In [8]:
# save the trained model
model.save('asl_model.keras')

In [26]:
# load the model and run real-time inference on webcam feed
# model = load_model('asl_model.keras')
cap = cv2.VideoCapture(0)

sequence = []
threshold = 0.8

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Failed to capture image.")
        break

    frame = cv2.flip(frame, 1)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5) as hands:
        
        results = hands.process(frame_rgb)

        left = np.zeros(63)
        right = np.zeros(63)

        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark]).flatten()
                if handedness.classification[0].label == 'Left':
                    left = coords
                else:
                    right = coords

            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        features = np.concatenate([left, right])
        sequence.append(features)
        sequence = sequence[-30:]

        if len(sequence) == 30:
            input_data = np.expand_dims(sequence, axis=0)
            prediction = model.predict(input_data, verbose=0)
            confidence = np.max(prediction)
            predicted_index = np.argmax(prediction)
            predicted_label = dynamic_label_names[predicted_index]

            # display on frame
            if confidence > threshold:
                cv2.putText(frame, f'{predicted_label} ({confidence:.2f})',
                        (10, 50), cv2.FONT_HERSHEY_SIMPLEX,
                        1.5, (0, 255, 0), 3)
            else:
                cv2.putText(frame, 'Unknown',
                        (10, 50), cv2.FONT_HERSHEY_SIMPLEX,
                        1.5, (0, 0, 255), 3)

    cv2.imshow('ASL Interpreter', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
